# H$_2$ VQE on Neutral-Atom Hardware

**Chemistry → Mapping → Noise → Validation** 📐

This notebook presents a minimal, application-facing workflow for mapping a small quantum chemistry problem to a neutral-atom-style execution pipeline. The emphasis is not only ideal-state optimization, but also identifying **where execution remains stable under noise and hardware-aware constraints**.


## 1. Imports

We use only lightweight scientific Python packages so the notebook stays easy to run and easy to understand.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt


## 2. Minimal H$_2$ Hamiltonian

For this first notebook, we use a compact 2-qubit toy Hamiltonian with coefficients often used in introductory H$_2$ demonstrations. This keeps the workflow focused on the full pipeline:

- chemistry target
- variational state
- neutral-atom-style mapping proxy
- noise sweep
- constraint-based validation


In [ ]:
H2_TERMS = [
    ("II", -1.0523732458),
    ("ZI",  0.3979374248),
    ("IZ", -0.3979374248),
    ("ZZ", -0.0112801043),
    ("XX",  0.1809311998),
]

H2_TERMS


## 3. Pauli operators and expectation values

We define a small amount of linear algebra to evaluate energies exactly for 2-qubit states.


In [ ]:
I = np.array([[1, 0], [0, 1]], dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

PAULI = {
    "I": I,
    "X": X,
    "Y": Y,
    "Z": Z,
}

def kron2(a, b):
    return np.kron(a, b)

def pauli_string_matrix(label: str) -> np.ndarray:
    if len(label) != 2:
        raise ValueError("This notebook expects 2-qubit Pauli strings.")
    return kron2(PAULI[label[0]], PAULI[label[1]])

def hamiltonian_matrix(terms) -> np.ndarray:
    H = np.zeros((4, 4), dtype=complex)
    for label, coeff in terms:
        H = H + coeff * pauli_string_matrix(label)
    return H

def expectation_value(state: np.ndarray, operator: np.ndarray) -> float:
    value = np.vdot(state, operator @ state)
    return float(np.real_if_close(value))

H = hamiltonian_matrix(H2_TERMS)
ground_energy = float(np.min(np.linalg.eigvalsh(H)))

print("Toy Hamiltonian ground-state energy:", ground_energy)
H


## 4. Variational ansatz

Instead of a single-angle state, we use a slightly richer hardware-friendly ansatz:

1. start from $|00\rangle$
2. apply single-qubit $R_Y$ rotations
3. apply a CNOT entangling gate
4. apply another layer of $R_Y$ rotations

This is still small enough for a first notebook, but expressive enough to produce a more meaningful optimization landscape.


In [ ]:
def ry(theta: float) -> np.ndarray:
    return np.array([
        [np.cos(theta / 2), -np.sin(theta / 2)],
        [np.sin(theta / 2),  np.cos(theta / 2)],
    ], dtype=complex)

def cnot_01() -> np.ndarray:
    return np.array([
        [1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 0, 1],
        [0, 0, 1, 0],
    ], dtype=complex)

def ansatz_state(params) -> np.ndarray:
    """Two-qubit hardware-friendly ansatz."""
    theta0, theta1, theta2, theta3 = params

    psi0 = np.array([1, 0, 0, 0], dtype=complex)
    U1 = np.kron(ry(theta0), ry(theta1))
    U2 = np.kron(ry(theta2), ry(theta3))
    Uent = cnot_01()

    psi = U2 @ (Uent @ (U1 @ psi0))
    psi = psi / np.linalg.norm(psi)
    return psi

params_demo = np.array([0.3, 0.8, 0.5, 0.2])
psi_demo = ansatz_state(params_demo)
psi_demo


## 5. Neutral-atom-style mapping proxy

This first notebook keeps the mapping layer simple. We define a small layout object with:

- number of atoms
- spacing
- blockade radius

This is not a full pulse-level compiler; it is a lightweight hardware-facing check that keeps the repo aligned with neutral-atom application workflows.


In [ ]:
class NeutralAtomLayout:
    def __init__(self, n_atoms: int, spacing_um: float, blockade_radius_um: float):
        self.n_atoms = n_atoms
        self.spacing_um = spacing_um
        self.blockade_radius_um = blockade_radius_um

    def blockade_active(self) -> bool:
        return self.spacing_um <= self.blockade_radius_um

    def summary(self) -> dict:
        return {
            "n_atoms": self.n_atoms,
            "spacing_um": self.spacing_um,
            "blockade_radius_um": self.blockade_radius_um,
            "blockade_active": self.blockade_active(),
        }

layout = NeutralAtomLayout(n_atoms=2, spacing_um=6.0, blockade_radius_um=7.0)
layout.summary()


## 6. Noise model

For this first notebook, we use a simple dephasing-style proxy instead of a full Lindblad solver. The idea is to preserve the state structure while damping non-reference amplitudes as noise increases.

This gives a clearer optimization story than collapsing everything directly to $|00\rangle$.


In [ ]:
REFERENCE_STATE = np.array([1.0, 0.0, 0.0, 0.0], dtype=complex)

def apply_noise_proxy(state: np.ndarray, gamma: float) -> np.ndarray:
    noisy = state.copy()
    noisy[1:] *= np.exp(-gamma)
    norm = np.linalg.norm(noisy)
    if norm == 0:
        raise ValueError("Noisy state norm is zero.")
    return noisy / norm

for g in [0.0, 0.25, 0.5, 1.0]:
    print(g, apply_noise_proxy(psi_demo, g))


## 7. Constraint-based validation 📐

We now define the validation layer. The notebook uses an overlap-based stability score against the reference state:

\[
\cos\theta = \frac{\langle r | \psi \rangle}{\|r\|\,\|\psi\|}
\]

with threshold

\[
\cos\theta \geq \frac{1}{\sqrt{1^2 + 1^2}}.
\]

This gives a compact constraint gate for deciding whether a state remains within a chosen stable execution region.


In [ ]:
THRESHOLD = 1.0 / np.sqrt(1.0**2 + 1.0**2)

def constraint_score(state: np.ndarray, reference: np.ndarray = REFERENCE_STATE) -> float:
    denom = np.linalg.norm(state) * np.linalg.norm(reference)
    if denom == 0:
        raise ValueError("Zero norm encountered in constraint_score.")
    value = np.vdot(reference, state) / denom
    return float(np.real_if_close(value))

def passes_constraint(state: np.ndarray, threshold: float = THRESHOLD) -> bool:
    return constraint_score(state) >= threshold

print("Threshold:", THRESHOLD)
print("Constraint score (demo state):", constraint_score(psi_demo))
print("Passes?", passes_constraint(psi_demo))


## 8. Energy evaluation and random-search sweep

We now scan noise level $\gamma$ and, for each noise value, search over a moderate number of random ansatz parameter sets.

For each sample we compute:

- noisy state
- energy
- constraint score
- pass/fail decision

This keeps the notebook simple and robust while still producing a useful application-facing result.


In [ ]:
def energy_of_state(state: np.ndarray) -> float:
    return expectation_value(state, H)

def evaluate_params_gamma(params, gamma: float) -> dict:
    state = ansatz_state(params)
    noisy_state = apply_noise_proxy(state, gamma)
    return {
        "params": np.array(params, dtype=float),
        "gamma": float(gamma),
        "energy": energy_of_state(noisy_state),
        "constraint_score": constraint_score(noisy_state),
        "passes": passes_constraint(noisy_state),
        "state": noisy_state,
    }

rng = np.random.default_rng(7)
gamma_grid = np.linspace(0.0, 1.25, 60)
n_samples = 4000

best_energy_all = []
best_params_all = []
best_score_all = []

best_energy_valid = []
best_params_valid = []
best_score_valid = []
valid_exists = []

for gamma in gamma_grid:
    sampled_params = rng.uniform(0.0, 2.0 * np.pi, size=(n_samples, 4))

    energies = []
    scores = []
    passes = []

    for params in sampled_params:
        result = evaluate_params_gamma(params, gamma)
        energies.append(result["energy"])
        scores.append(result["constraint_score"])
        passes.append(result["passes"])

    energies = np.array(energies)
    scores = np.array(scores)
    passes = np.array(passes, dtype=bool)

    idx_all = int(np.argmin(energies))
    best_energy_all.append(energies[idx_all])
    best_params_all.append(sampled_params[idx_all])
    best_score_all.append(scores[idx_all])

    if np.any(passes):
        valid_energies = np.where(passes, energies, np.inf)
        idx_valid = int(np.argmin(valid_energies))
        best_energy_valid.append(energies[idx_valid])
        best_params_valid.append(sampled_params[idx_valid])
        best_score_valid.append(scores[idx_valid])
        valid_exists.append(True)
    else:
        best_energy_valid.append(np.nan)
        best_params_valid.append(np.array([np.nan] * 4))
        best_score_valid.append(np.nan)
        valid_exists.append(False)

best_energy_all = np.array(best_energy_all)
best_energy_valid = np.array(best_energy_valid)
best_params_all = np.array(best_params_all)
best_params_valid = np.array(best_params_valid)
best_score_all = np.array(best_score_all)
best_score_valid = np.array(best_score_valid)
valid_exists = np.array(valid_exists, dtype=bool)

print("Number of gamma values with at least one constraint-passing sample:", int(np.sum(valid_exists)), "of", len(gamma_grid))


## 9. Dense pass/fail map over $(\gamma, \theta_0)$

For visualization, we also construct a smaller structured slice through parameter space. We vary $\theta_0$ while holding the other angles fixed, then evaluate the constraint gate over noise.

This gives a readable picture of the stable execution region without needing to visualize the full 4-parameter landscape.


In [ ]:
theta0_grid = np.linspace(0.0, 2.0 * np.pi, 160)
slice_params = np.array([0.0, 1.0, 0.8, 0.5])
pass_surface = np.zeros((len(gamma_grid), len(theta0_grid)), dtype=bool)
score_surface = np.zeros((len(gamma_grid), len(theta0_grid)))
energy_surface = np.zeros((len(gamma_grid), len(theta0_grid)))

for i, gamma in enumerate(gamma_grid):
    for j, theta0 in enumerate(theta0_grid):
        params = slice_params.copy()
        params[0] = theta0
        result = evaluate_params_gamma(params, gamma)
        pass_surface[i, j] = result["passes"]
        score_surface[i, j] = result["constraint_score"]
        energy_surface[i, j] = result["energy"]


## 10. Plot: best energy under noise

This plot gives the main application-facing summary for Notebook 01.


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(gamma_grid, best_energy_all, label="Best energy (all states)")
plt.plot(gamma_grid, best_energy_valid, label="Best energy (constraint-passing)")
plt.axhline(ground_energy, linestyle="--", label="Ground-state energy of toy Hamiltonian")
plt.xlabel("Noise γ")
plt.ylabel("Energy")
plt.title("H₂ VQE under noise with constraint-based validation")
plt.legend()
plt.tight_layout()
plt.show()


## 11. Plot: stable execution region under constraint gate

This figure shows a readable parameter slice indicating which regions remain inside the chosen validation threshold.


In [ ]:
plt.figure(figsize=(8, 5))
plt.imshow(
    pass_surface.astype(float),
    aspect="auto",
    origin="lower",
    extent=[theta0_grid.min(), theta0_grid.max(), gamma_grid.min(), gamma_grid.max()],
)
plt.colorbar(label="Constraint pass (1=yes, 0=no)")
plt.xlabel("θ₀")
plt.ylabel("Noise γ")
plt.title("Stable execution region under overlap-based constraint gate")
plt.tight_layout()
plt.show()


## 12. Plot: best variational parameters under noise

Since the ansatz now has four angles, we plot two representative optimized parameters.


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(gamma_grid, best_params_all[:, 0], label="Best θ₀ (all states)")
plt.plot(gamma_grid, best_params_all[:, 1], label="Best θ₁ (all states)")
plt.xlabel("Noise γ")
plt.ylabel("Optimal angle")
plt.title("Best variational parameters under noise")
plt.legend()
plt.tight_layout()
plt.show()


## 13. Plot: best constraint score under noise

This shows how the best overlap score changes as noise increases.


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(gamma_grid, best_score_all, label="Best-score energy state's constraint score")
plt.plot(gamma_grid, best_score_valid, label="Best valid state's constraint score")
plt.axhline(THRESHOLD, linestyle="--", label="Constraint threshold")
plt.xlabel("Noise γ")
plt.ylabel("Constraint score")
plt.title("Constraint score under noise")
plt.legend()
plt.tight_layout()
plt.show()


## 14. Save a figure for the repo

This cell saves a clean figure to `../figures/` when run from the `notebooks/` directory inside the repo. It also creates the directory automatically if needed.


In [ ]:
output_dir = os.path.join("..", "figures")
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "h2_energy_vs_noise.png")

plt.figure(figsize=(8, 5))
plt.plot(gamma_grid, best_energy_all, label="Best energy (all states)")
plt.plot(gamma_grid, best_energy_valid, label="Best energy (constraint-passing)")
plt.axhline(ground_energy, linestyle="--", label="Ground-state energy of toy Hamiltonian")
plt.xlabel("Noise γ")
plt.ylabel("Energy")
plt.title("H₂ VQE under noise with constraint-based validation")
plt.legend()
plt.tight_layout()
plt.savefig(output_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved figure to: {output_path}")


## 15. Summary

This revised first notebook establishes a minimal application-facing pipeline:

- define a chemistry target
- construct a richer variational state
- include a neutral-atom-style mapping proxy
- sweep noise
- validate execution with a constraint gate

Compared with a single-angle toy ansatz, this version gives a more meaningful optimization landscape and more informative application-level plots.


## Notes

This workflow can also be read as a three-stage pipeline:

- algorithm definition
- hardware mapping
- execution validation

That structure generalizes beyond this notebook to broader quantum applications work.
